In [1]:
import pandas as pd
import ast
import glob
import os

# Aumento da base de dados

Como baseline temos a base com sentenças, que são todas regras que são utilizadas no pipeline do ACCORD-NLP.

A fim obter exemplos negativos, ou seja não regras, utilizamos a base de dados do CODE-ACCORD, o qual retirou as sentenças de PDF de regulamentações filandesas e inglesas.
A parti disso, foi feita uma anotação na mão de quais são regras e quais não são. As demais forma aramazenadas em tabelas.

Foi utilizado essas tabelas para unir em uma unica só, para obter exemplos de ambas classes. 

In [2]:
df_sentenses_baseline = pd.read_csv('../data/all.csv')

In [3]:
# Diretórios onde estão as tabelas a serem unidas
paths = [
    "../data/8-Classification",
    "../data/CSV-All-Semi-Filtered-Sentences-AllChapters"
 ]

all_dfs = []

for path in paths:
    # Busca recursiva por todos os arquivos .csv e .xlsx
    csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
    xlsx_files = glob.glob(os.path.join(path, "**", "*.xlsx"), recursive=True)
    files = csv_files + xlsx_files
    for f in files:
        if f.endswith(".xlsx"):
            df = pd.read_excel(f)
        else:
            df = pd.read_csv(f)
        df["source_file"] = os.path.basename(f)
        all_dfs.append(df)

# Concatena tudo em um único DataFrame
full_df = pd.concat(all_dfs, ignore_index=True)
print(full_df.columns)
if "label" in full_df.columns:
    print(full_df["label"].value_counts())

Index(['ID', 'Text', 'Unnamed: 2', 'source_file', 'Class', 'Unnamed: 3',
       'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8',
       'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12',
       'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16',
       'Unnamed: 17', 'Unnamed: 18', 'Other', 'Unnamed: 19', 'Unnamed: 20',
       'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24',
       'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28',
       'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32',
       'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36',
       'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40',
       'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44',
       'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48',
       'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52',
       'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56',
       'Unnamed: 57', 'Unnamed: 58', '

In [4]:
full_df.isnull().sum(), full_df.shape

(ID                  15
 Text                14
 Unnamed: 2        6075
 source_file          0
 Class             2655
                   ... 
 other             6076
 self-contained    5819
 class             6048
 Class             5898
 class             6002
 Length: 79, dtype: int64,
 (6077, 79))

In [5]:
full_df_filted = full_df[['ID', 'Text', 'source_file', 'Class', 'Other', 'other', 'self-contained', 'class ',
       'Class ', 'class']].copy()
full_df_filted.head()

,ID,Text,source_file,Class,Other,other,self-contained,class,Class,class
0,3_UK_DocD_ToxicSubstances,If insulating material is inserted into a cavi...,UK-All-self-contained-Sentences.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4_UK_DocD_ToxicSubstances,To reduce the risks to the health of persons i...,UK-All-self-contained-Sentences.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9_UK_DocD_ToxicSubstances,If insulating material is inserted into a cavi...,UK-All-self-contained-Sentences.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12_UK_DocD_ToxicSubstances,To reduce the risks to the health of persons i...,UK-All-self-contained-Sentences.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,14_UK_DocD_ToxicSubstances,Insulating materials which give off formaldehy...,UK-All-self-contained-Sentences.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
TEXT_CLASS_COLS  = ['Class', 'class', 'Class ', 'class ']   # contêm o valor como texto
FLAG_SC_COLS     = ['self-contained']                        # flag: presença = self-contained
FLAG_OT_COLS     = ['Other', 'other']  
PATH_CLASS_DIR  = '../data/8-Classification'
PATH_SEMI_DIR   = '../data/CSV-All-Semi-Filtered-Sentences-AllChapters'

In [6]:
def resolve_class(row, available_cols):
    for col in TEXT_CLASS_COLS:
        if col in available_cols:
            val = row.get(col)
            if pd.notna(val) and str(val).strip() != '':
                val_norm = str(val).strip().lower()
                if val_norm == 'self-contained':
                    return 'self-contained'
                elif val_norm in ['others', 'other']:
                    return 'others'
 
    # 2. Flags booleanas — a presença de qualquer valor indica a classe
    for col in FLAG_SC_COLS:
        if col in available_cols:
            val = row.get(col)
            if pd.notna(val) and str(val).strip() not in ['', 'nan', 'None']:
                return 'self-contained'
 
    for col in FLAG_OT_COLS:
        if col in available_cols:
            val = row.get(col)
            if pd.notna(val) and str(val).strip() not in ['', 'nan', 'None']:
                return 'others'
 
    return None

In [11]:
class_dfs = []
files_class = (glob.glob(os.path.join(PATH_CLASS_DIR, '**', '*.csv'),  recursive=True) +
               glob.glob(os.path.join(PATH_CLASS_DIR, '**', '*.xlsx'), recursive=True))
 
for f in files_class:
    if f.endswith(".xlsx"):
            df_f = pd.read_excel(f)
    else:
            df_f = pd.read_csv(f)
    # df_f = read_file(f)
    avail = df_f.columns.tolist()
 
    # Garante coluna Text
    for tc in ['Text', 'text', 'Sentence', 'sentence', 'content']:
        if tc in avail:
            df_f = df_f.rename(columns={tc: 'Text'})
            break
 
    # Garante coluna ID
    for ic in ['ID', 'id', 'Id']:
        if ic in avail and ic != 'ID':
            df_f = df_f.rename(columns={ic: 'ID'})
            break
    if 'ID' not in df_f.columns:
        df_f['ID'] = None
 
    # Resolve classe linha a linha usando TODAS as colunas disponíveis
    df_f['class_resolved'] = df_f.apply(
        lambda row: resolve_class(row, df_f.columns.tolist()), axis=1
    )
    df_f['source'] = 'classification'
 
    # is_rule: 1=self-contained, 0=others, NaN=desconhecido
    df_f['is_rule'] = df_f['class_resolved'].map(
        {'self-contained': 1, 'others': 0}
    )
 
    class_dfs.append(df_f[['Text', 'is_rule', 'ID', 'source', 'class_resolved']])

In [12]:
if class_dfs:
    df_class = pd.concat(class_dfs, ignore_index=True)
    print(f"    {len(df_class)} linhas brutas de {len(files_class)} arquivos")
    print(f"    class_resolved nulos: {df_class['class_resolved'].isna().sum()}")
    print(f"    is_rule=1: {(df_class['is_rule']==1).sum()} | "
          f"is_rule=0: {(df_class['is_rule']==0).sum()} | "
          f"is_rule=NaN: {df_class['is_rule'].isna().sum()}")
else:
    df_class = pd.DataFrame(columns=['Text','is_rule','ID','source','class_resolved'])
    print("    Nenhum arquivo encontrado.")

    4685 linhas brutas de 21 arquivos
    class_resolved nulos: 2098
    is_rule=1: 797 | is_rule=0: 1790 | is_rule=NaN: 2098


In [13]:
df_class['is_rule'].unique(), df_class.shape

(array([nan,  1.,  0.]), (4685, 5))

In [45]:
df_sentenses_baseline['is_rule'] = 1
df_sentenses_baseline['source'] = 'baseline'
df_sentenses_baseline = df_sentenses_baseline.rename(columns={'content': 'Text'})
df_sentenses_baseline['class_resolved'] = 'self-contained'
df_sentenses_baseline.to_csv('../data/all.csv', index=False)

full_df_final = pd.concat([df_class, df_sentenses_baseline[['Text','is_rule','metadata','class_resolved', 'source']]], ignore_index=True)
full_df_final.head()
print(full_df_final.shape)
full_df_final.columns

(5547, 6)


Index(['Text', 'is_rule', 'ID', 'source', 'class_resolved', 'metadata'], dtype='object')

In [15]:
full_df_final.isnull().sum()

Text                14
is_rule           2098
ID                 876
source               0
class_resolved    2098
metadata          4685
dtype: int64

In [16]:
full_df_final.to_csv('../data/full_dataset.csv', index=False)

# Limpeza da base de dados

In [17]:
# df = pd.read_csv('../data/classification_dataset_augmented.csv')
df = pd.read_csv('../data/full_dataset.csv')


## Base de dados

Essa base foi contruida com a junção de todas as tabelas que contiam sentenças, integrando 34 tabelas.
Juntando um unica base de dados com 6937 instancias

In [18]:
print("Quantidade de linhas e colunas:", df.shape)
df.head()

Quantidade de linhas e colunas: (5547, 6)


,Text,is_rule,ID,source,class_resolved,metadata
0,If insulating material is inserted into a cavi...,NaN,3_UK_DocD_ToxicSubstances,classification,NaN,NaN
1,To reduce the risks to the health of persons i...,NaN,4_UK_DocD_ToxicSubstances,classification,NaN,NaN
2,If insulating material is inserted into a cavi...,NaN,9_UK_DocD_ToxicSubstances,classification,NaN,NaN
3,To reduce the risks to the health of persons i...,NaN,12_UK_DocD_ToxicSubstances,classification,NaN,NaN
4,Insulating materials which give off formaldehy...,NaN,14_UK_DocD_ToxicSubstances,classification,NaN,NaN


A coluna metadata e ID tem a mesma finalidade, ser um identificados de onde foi retirado a sentença e o documento.


In [19]:
df.isnull().sum()

Text                14
is_rule           2098
ID                 876
source               0
class_resolved    2098
metadata          4685
dtype: int64

## Padronização de nomenclaturas e tipos de dados

In [20]:
fil = df['metadata'].notnull()
df_copy = df[fil].copy()
for i in df_copy['metadata'].values:
    print(i[5:100])

: '69_Finnish_FireSafety'}
: '62_Finnish_Accessibility'}
: '33_Finnish_Accessibility'}
: '107_Finnish_FireSafety'}
: '16_Finnish_Accessibility'}
: '17_Finnish_AcousticEnvironment'}
: '73_Finnish_FireSafety'}
: '105_Finnish_EnergyEfficiency'}
: '67_Finnish_Accessibility'}
: '130_Finnish_FireSafety'}
: '29_Finnish_Accessibility'}
: '26_Finnish_AcousticEnvironment'}
: '27_Finnish_Accessibility'}
: '42_Finnish_Accessibility'}
: '258_Finnish_FireSafety'}
: '121_Finnish_FireSafety'}
: '26_Finnish_FireSafety'}
: '67_Finnish_EnergyEfficiency'}
: '90_Finnish_FireSafety'}
: '57_Finnish_Accessibility'}
: '93_Finnish_Health-WaterAndSewerage'}
: '181_Finnish_FireSafety'}
: '80_Finnish_Health-Humidity'}
: '189_Finnish_FireSafety'}
: '241_Finnish_FireSafety'}
: '222_Finnish_FireSafety'}
: '133_Finnish_Health-WaterAndSewerage'}
: '125_Finnish_Health-WaterAndSewerage'}
: '36_Finnish_Health-Humidity'}
: '217_Finnish_FireSafety'}
: '240_Finnish_FireSafety'}
: '59_Finnish_Health-WaterAndSewerage'}
: '135_

In [21]:
def extract_id_from_metadata(val):
    if pd.isna(val): # Se for vazio, ignora
        return None
    try:
        # ast.literal_eval transforma a string "{'ID': '123'}" num dicionário real do Python
        d = ast.literal_eval(val)
        if isinstance(d, dict) and 'ID' in d:
            return d['ID']
    except:
        pass
    return None

In [22]:
df['metadata_id'] = df['metadata'].apply(extract_id_from_metadata)
df['ID'] = df['ID'].fillna(df['metadata_id'])

In [23]:
df_id_clear = df.drop(columns=['metadata', 'metadata_id'])
df_id_clear

,Text,is_rule,ID,source,class_resolved
0,If insulating material is inserted into a cavi...,NaN,3_UK_DocD_ToxicSubstances,classification,NaN
1,To reduce the risks to the health of persons i...,NaN,4_UK_DocD_ToxicSubstances,classification,NaN
2,If insulating material is inserted into a cavi...,NaN,9_UK_DocD_ToxicSubstances,classification,NaN
3,To reduce the risks to the health of persons i...,NaN,12_UK_DocD_ToxicSubstances,classification,NaN
4,Insulating materials which give off formaldehy...,NaN,14_UK_DocD_ToxicSubstances,classification,NaN
...,...,...,...,...,...
5542,The riser of the steps in an exit may be no mo...,1.0,25_Finnish_SafetyOfUse,baseline,self-contained
5543,A landing measuring at least 800 millimetres i...,1.0,34_Finnish_SafetyOfUse,baseline,self-contained
5544,Water pipes and sewers installed in the ground...,1.0,146_Finnish_Health-WaterAndSewerage,baseline,self-contained
5545,A cube with edges of no more than 200 millimet...,1.0,50_Finnish_SafetyOfUse,baseline,self-contained


## Sentenças nulas

In [24]:
df_id_clear = df_id_clear.dropna(subset=['Text'])
df_id_clear

,Text,is_rule,ID,source,class_resolved
0,If insulating material is inserted into a cavi...,NaN,3_UK_DocD_ToxicSubstances,classification,NaN
1,To reduce the risks to the health of persons i...,NaN,4_UK_DocD_ToxicSubstances,classification,NaN
2,If insulating material is inserted into a cavi...,NaN,9_UK_DocD_ToxicSubstances,classification,NaN
3,To reduce the risks to the health of persons i...,NaN,12_UK_DocD_ToxicSubstances,classification,NaN
4,Insulating materials which give off formaldehy...,NaN,14_UK_DocD_ToxicSubstances,classification,NaN
...,...,...,...,...,...
5542,The riser of the steps in an exit may be no mo...,1.0,25_Finnish_SafetyOfUse,baseline,self-contained
5543,A landing measuring at least 800 millimetres i...,1.0,34_Finnish_SafetyOfUse,baseline,self-contained
5544,Water pipes and sewers installed in the ground...,1.0,146_Finnish_Health-WaterAndSewerage,baseline,self-contained
5545,A cube with edges of no more than 200 millimet...,1.0,50_Finnish_SafetyOfUse,baseline,self-contained


## Remoção de duplicados

In [25]:
df_clean = df_id_clear.groupby('Text', as_index=False).agg({
    'is_rule': 'max',  # Prioriza 1 em caso de conflito entre 0 e 1
    
    # Junta os IDs diferentes numa única string, ignorando valores vazios e repetidos
    'ID': lambda x: ' | '.join(sorted(set([str(i) for i in x if pd.notna(i)])))
})
df_clean

,Text,is_rule,ID
0,A vent to the outside with a minimum free are...,NaN,155_UK_DocB_V1_FireSafety
1,"As an alternative to paragraph 3.17, to venti...",0.0,153_UK_DocF_V1_Ventilation
2,Duct connections should be both mechanically ...,1.0,116_UK_DocF_V1_Ventilation
3,For fixed terminals with flow adjustment by d...,0.0,204_UK_DocF_V1_Ventilation
4,Hinged or pivot windows with an opening angle...,NaN,53_UK_DocF_V1_Ventilation
...,...,...,...
2763,there is no expectation that lighting calculat...,0.0,311_UK_DocM_V2_AccessAndUseOfBuildings
2764,this may be the case the Construction (Design ...,0.0,12_UK_DocK_ProtectionFromFalling
2765,totalling up to 50mm total thickness and shall...,1.0,131_UK_DocM_V1_AccessAndUseOfBuildings
2766,where there is insufficient space for larger r...,1.0,239_UK_DocL_V1_ConsrvationOfFuelAndPower


In [26]:
df_clean.duplicated().sum(), df_clean.isnull().sum()

(np.int64(0),
 Text         0
 is_rule    150
 ID           0
 dtype: int64)

In [27]:
df_clean.value_counts('is_rule')

is_rule
0.0    1578
1.0    1040
Name: count, dtype: int64

## Tratamento de regras sem classificação

Baseado no artigo do CODE-ACCORD, define três critérios simultâneos para uma sentença ser self-contained. Para ser regra, ela precisa passar nos três:

* Completude normativa: expressa uma obrigação, proibição ou permissão completa. Operacionalmente: contém pelo menos um marcador deôntico (shall, must, should, may, must not, shall not).

* Sem co-referências não resolvíveis: não depende de contexto externo para ser compreendida. Operacionalmente: não começa com pronome sem antecedente claro (It, They, This, These, Such) nem contém the above, the following, as specified.

* Sem referências externas: não aponta para seções, tabelas, figuras ou documentos externos. Operacionalmente: não contém Section, Table, Figure, Appendix, Clause, Part, see, refer to.

In [28]:
import re

def classify_by_article_criteria(text):
    if pd.isna(text) or str(text).strip() == '':
        return None
    
    t = str(text).strip()
    t_lower = t.lower()
    
    # CRITÉRIO 2 — Co-referência: descarta imediatamente (is_rule = 0)
    coref_patterns = [
        r'^(it|they|this|these|such|those|that)\b',
        r'\bthe above\b', r'\bthe following\b',
        r'\bas specified\b', r'\bas described\b',
        r'\bas mentioned\b', r'\breferred to above\b'
    ]
    for pat in coref_patterns:
        if re.search(pat, t_lower):
            return 0
    
    # CRITÉRIO 3 — Referência externa: descarta (is_rule = 0)
    external_ref = [
        r'\bsection\b', r'\btable\b', r'\bfigure\b',
        r'\bappendix\b', r'\bclause\b', r'\bpart [a-z0-9]\b',
        r'\bsee\b', r'\brefer to\b', r'\bparagraph\b',
        r'\bschedule\b', r'\bannex\b'
    ]
    for pat in external_ref:
        if re.search(pat, t_lower):
            return 0
    
    # CRITÉRIO 1 — Marcador deôntico: confirma (is_rule = 1)
    deontic = [
        r'\bshall\b', r'\bmust\b', r'\bshould\b',
        r'\bmay not\b', r'\bmust not\b', r'\bshall not\b',
        r'\bis required\b', r'\bare required\b',
        r'\bis prohibited\b', r'\bshall be\b'
    ]
    for pat in deontic:
        if re.search(pat, t_lower):
            return 1
    
    # Passou nos critérios 2 e 3 mas sem marcador deôntico → não é regra
    return 0


In [29]:
# df_clean
mask_nan = df_clean['is_rule'].isna()
df_clean.loc[mask_nan, 'is_rule'] = df_clean.loc[mask_nan, 'Text'].apply(
    classify_by_article_criteria
)

print(f'NaN restantes: {df_clean["is_rule"].isna().sum()}')
print(df_clean['is_rule'].value_counts(dropna=False))

NaN restantes: 0
is_rule
0.0    1588
1.0    1180
Name: count, dtype: int64


## Salvar nova base de dados tratado

In [30]:
df_clean.to_csv('../data/classification_dataset_cleaned(2).csv', index=False)

#

# Limpeza na base de dados ACCORD

Como é possivel ver, essa base ue o artigo disponibiliza tem valores repetidos, os quais não são levados em consideração quando foi realyzado nos treinamantos dos modelos.
A consluão que podemos tomar é que os resultados chegado são erroneis, pois nçao tratam dessa caracetristicas, apeas é um split de teste.

In [7]:
df_base = pd.read_csv('../data/Single-Clauses-Data_Binary-Classification.csv', encoding='latin1')
df_base.shape

/tmp/ipykernel_15157/3729387904.py:1: DtypeWarning: Columns (20,21,72) have mixed types. Specify dtype option on import or set low_memory=False.
  df_base = pd.read_csv('../data/Single-Clauses-Data_Binary-Classification.csv', encoding='latin1')


(25867, 73)

In [8]:
df_base.isnull().sum()

1_UK_DocG_Sanitation                                                                                  10
Others                                                                                                 0
However, building work may be subject to more than one requirement of the Building Regulations.        0
Unnamed: 3                                                                                         25867
Unnamed: 4                                                                                         25867
                                                                                                   ...  
Unnamed: 68                                                                                        25867
Unnamed: 69                                                                                        25867
Unnamed: 70                                                                                        25867
Unnamed: 71                                            

In [32]:
df_base.columns

Index(['1_UK_DocG_Sanitation', 'Others',
       'However, building work may be subject to more than one requirement of the Building Regulations.',
       'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7',
       'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12',
       'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16',
       'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'self-contained',
       'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24',
       'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28',
       'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32',
       'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36',
       'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40',
       'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44',
       'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48',
       'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52',
       'Unnamed: 53', '

In [33]:
df_base.duplicated().sum()

np.int64(23276)

In [10]:
df_duplicated = df_base[df_base['However, building work may be subject to more than one requirement of the Building Regulations.'].duplicated()]
df_duplicated.iloc[78]['However, building work may be subject to more than one requirement of the Building Regulations.']

'Where modelling or monitoring data is required, expert advice should be sought.'

In [11]:
f = df_base['However, building work may be subject to more than one requirement of the Building Regulations.'] == 'However, building work may be subject to more than one requirement of the Building Regulations.'
df_base[f]

,1_UK_DocG_Sanitation,Others,"However, building work may be subject to more than one requirement of the Building Regulations.",Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 63,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67,Unnamed: 68,Unnamed: 69,Unnamed: 70,Unnamed: 71,Unnamed: 72
2581,1_UK_DocG_Sanitation,Others,"However, building work may be subject to more ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5169,1_UK_DocG_Sanitation,Others,"However, building work may be subject to more ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7756,1_UK_DocG_Sanitation,Others,"However, building work may be subject to more ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10343,1_UK_DocG_Sanitation,Others,"However, building work may be subject to more ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12930,1_UK_DocG_Sanitation,Others,"However, building work may be subject to more ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15517,1_UK_DocG_Sanitation,Others,"However, building work may be subject to more ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18103,1_UK_DocG_Sanitation,Others,"However, building work may be subject to more ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20690,1_UK_DocG_Sanitation,Others,"However, building work may be subject to more ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23279,1_UK_DocG_Sanitation,Others,"However, building work may be subject to more ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
## Tratamento
remove_unname_columns = [col for col in df_base.columns if col.startswith('Unnamed')]

In [13]:
df_base_clear = df_base.drop(columns=remove_unname_columns)
df_base_clear.columns

Index(['1_UK_DocG_Sanitation', 'Others',
       'However, building work may be subject to more than one requirement of the Building Regulations.',
       'self-contained'],
      dtype='object')

In [14]:
df_base_clear.tail()

,1_UK_DocG_Sanitation,Others,"However, building work may be subject to more than one requirement of the Building Regulations.",self-contained
25862,500_UK_DocM_V2_AccessAndUseOfBuildings,Others,Any general lighting system within the area se...,NaN
25863,501_UK_DocM_V2_AccessAndUseOfBuildings,Others,Appendix E: Hierarchy for establishing seasona...,NaN
25864,502_UK_DocM_V2_AccessAndUseOfBuildings,Others,E1 When a heating system is being replaced in...,NaN
25865,503_UK_DocM_V2_AccessAndUseOfBuildings,Others,The seasonal efficiency of the appliance being...,NaN
25866,504_UK_DocM_V2_AccessAndUseOfBuildings,Others,5) should be made to convert it to an appropri...,NaN


In [15]:
df_base_clear.isnull().sum()

1_UK_DocG_Sanitation                                                                                  10
Others                                                                                                 0
However, building work may be subject to more than one requirement of the Building Regulations.        0
self-contained                                                                                     25818
dtype: int64

In [16]:
df_base_clear.shape

(25867, 4)

## Padronização das colunas das bases

In [ ]:
df_sentenses_baseline['']
df_sentenses_baseline.columns

Index(['example_id', 'Text', 'processed_content', 'label', 'metadata',
       'is_rule', 'source', 'class_resolved'],
      dtype='object')